# Investigate the best value for a cut on the Double charm BDT_all

### Setting up the data and tools

In [4]:
# Importing the notebook with common setup 
%run 'setup_mc.ipynb'

Welcome to JupyROOT 6.28/00


Invoke (root_datasets, pandas_datasets) = load_data(inclmc_type="23903000") to load datasets
Invoke (root_datasets, pandas_datasets) = load_data(inclmc_type="23903003") to load dataset for double charm
Invoke  df_signal_23903000 = load_signal_from_inclMC() to load signal from 23903000 Inclusive MC
or
Invoke  df_signal = load_signal_all()
Invoke  df_background = load_background_category(category)


In [5]:
import xgboost as xgb
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import RocCurveDisplay, accuracy_score
from sklearn.metrics import roc_curve, auc
from sklearn.utils import shuffle

## Loading the BDTs

In [6]:
import joblib
bdt_all = joblib.load("../bdtdblcharm_150_3_0.04.pkl")

## Loading the data

In [7]:
dfall = load_complete_df()
dfall.shape

(649799, 36)

In [8]:
# We check the BDTs on eeventIndex 1 as they were traineds on eventIndex 0
df = dfall.query("eventIndex == 1")
dfall = df
df.shape

(324816, 36)

## Adding columns with the output of the BDTs for categories all

In [9]:
#df['category']
#mygroupby(df, 'category')
df = df.copy()
train_columns = [
    "Y_0_40_nc_mult",
    "Y_0_20_cc_mult",
    "Y_0_20_cc_PZ",
    "Y_0_30_nc_PZ",
    "Y_0_40_nc_PZ",
    "min_m2pi",
    "max_m2pi",
    "missing_mass_2",
    "B_BPVVDR",
    "B_M",
    "B_correctedMass",
    "log(abs(PBsn))",
    "log(abs(PBv/B_P))",
    "log(abs(PBvn/B_P))",
    "log(abs((PBsn-PBvn)/PBvn))",
    "log(sqrt(abs(mDs2vn)))",
    "mN2v",
    "log(Y_PE)",
    "BDT_Iso",
    "B_pT_Bdir",
    "Y_BPVVDR",
    "missing_pY_mass",
    "Y_correctedMass",
]
pred = bdt_all.predict_proba(df[train_columns])
df['bdt_all'] = pred[:,1]
df['bdt_dc'] = pred[:,1]

In [10]:
mycategs = [18,29,23,15,0,16,14,25,7,19,13,27,22,20,26,21]

In [11]:
others = set(df.category) - set(mycategs)
".".join([str(c) for c in others])

'1.2.3.4.6.8.9.10.11.12.17.24.28.30.31.32.33.34.35.36.37.38.39.40.41.43.44.45.46.47.48.-1'

In [12]:
df.query("bdt_all > 0.75").query("category == 18 or category == 19")['category']

274       19
325       19
355       18
390       19
392       19
          ..
336532    18
336836    19
337183    19
337482    19
337512    19
Name: category, Length: 5302, dtype: int32

## Tool to find templates with similar projections using Kolmogorov Smirnov

In [13]:
# from scipy.stats import ks_2samp

# def similar_categories(df, shown_number=17):
#     """ Finding which categories have similar histograms for q2_2, tauY_2, bdt_all """
    
#     # Grouping the samples per category
#     c = mygroupby(df, 'category')
#     c['name'] = c.apply(lambda row: categories[f"{row['category']:.4g}"], axis=1)
    
#     # Preparing the list of categories to process
#     #shown_number=17
#     shown_categs = list(c.head(shown_number)['category'])

#     # Building a list of detasets filtered 
#     datasets = { f"{c}":  df.query(f"category == {c}") for c in shown_categs }
#     datasets["others"] =  df.query(f"category not in {shown_categs}")
#     datasets_names = [ f"{c}:" + categories[f"{c}"] for c in shown_categs]
#     datasets_names.append("others")
    
#     # List of column names
#     cols = np.array([ str(c) for c in shown_categs ] + [ "others" ])

#     # Preparing the matrix with the final results
#     stats= pd.DataFrame(np.zeros((shown_number +1, shown_number + 1)), columns=cols)
#     stats["name"] = cols.T
#     stats = stats.set_index("name", drop=True).copy()

#     stats_q2_2 = stats.copy()
#     stats_tauY_2 = stats.copy()
#     stats_bdt_all = stats.copy()
    
#     for i in range(len(cols)):
#         for j in range(len(cols)):
#             # Only deal with half the matrix
#             if j > i:
#                 continue
            
#             todo = { "q2_2": stats_q2_2,
#                      "tauY_2": stats_tauY_2,
#                      "bdt_all": stats_bdt_all,
#                    }
#             for myvar, matrix in todo.items():
#                 res = ks_2samp(datasets[cols[i]][myvar], datasets[cols[j]][myvar], alternative='two-sided', method='exact')
#                 matrix[cols[i]][cols[j]] = res[1]
#                 matrix[cols[j]][cols[i]] = matrix[cols[i]][cols[j]] 
#     return stats_q2_2, stats_tauY_2, stats_bdt_all

## Tool to analyze matrix returned by "similar_categories" and group the templates

In [14]:
# def find_clusters(m, threshold):
#     allsets = set()
#     for i in m.index:
#         s = frozenset(m[m[i] > threshold][i].index)
#         if s:
#             allsets.add(s)
#     return allsets

# # Iterative
# def merge_clusters(clusters):
#     final = []
#     tomerge = list(clusters)
#     while True:
#         h = tomerge[0]
#         t = tomerge[1:]
#         if not t:
#             final.append(h)
#             break
#         tmp = []
#         merged = False
#         for c in t:
#             if h & c:
#                 tmp.append( h | c)
#                 merged = True
#             else:
#                 tmp.append(c)
#         if not merged:
#             final.append(h)
#         tomerge = tmp
#     return final

# def merge_clusters_rec(unmerged, merged):
#     """ Merge sets with non null intersection """
#     h = unmerged[0]
#     t = unmerged[1:]
    
#     # End of recursion if ony one element in the list,nothing to merge to....
#     if not t:
#         merged.append(h)
#         return t, merged
    
#     # Iterate over the list of elements, comparing the first one h,
#     # with all the others. If h has no intersection we add it to the
#     # list of processed elements (merged), otherwise we merge it to the
#     # element in common and we iterate the function again on the 
#     # new list containing this new element plus the others
#     tmp = []
#     was_merged = False
#     for c in t:
#         if h & c:
#             tmp.append( h | c)
#             was_merged = True
#         else:
#             tmp.append(c)
#     if not was_merged:
#         merged.append(h)

#     return merge_clusters_rec(tmp, merged)


# def find_and_merge_clusters(m, threshold=0.5):
#     clusters = find_clusters(m, threshold)
#     unmerged, merged = merge_clusters_rec(list(clusters), [])
#     return [ list(s) for s in merged ]
